# Notebook 5: Long-Term Memory — EPISODIC (Past Experiences)

### What it does (simplest version)
Remembers **complete past episodes**: what happened → what the agent did → how it turned out.
When a similar situation appears, the agent **recalls the episode and repeats what worked**.

### The difference from Notebook 4
| Semantic (NB4) | Episodic (NB5) |
|---|---|
| *What the agent **knows*** | *What the agent has **done*** |
| Single facts: "likes biryani" | Whole stories: "angry customer → apologized + coupon → 5 stars" |
| Injected as background info | Injected as **few-shot examples** |

### Human analogy
A senior support agent: *"Last time a ticket like this came in, here's what worked."*
That sentence — recalled at the right moment — is episodic memory.

### The new trick: semantic search
In Notebook 4 we listed **all** facts in a namespace. Episodes need something smarter:
find episodes that are **similar in meaning** to the current situation — even when no
words match exactly. That's done with **embeddings** (the same idea behind RAG).

Think of it as: **RAG over the agent's own history.**

### Where to use it
Support bots (recall similar resolved tickets), coding agents (recall how a similar bug
was fixed), any agent you want to **improve from its own track record**.

### What it connects with later
Episodic memory copies *successful actions*. But sometimes you want to change
**the rules themselves** — how the agent behaves in general. That's Notebook 6: procedural memory.

## Step 1 — Setup (note the new `index=` line)

`index={"dims": 1536, "embed": "openai:text-embedding-3-small"}` turns on
**semantic search** for this store: every value is converted to an embedding vector,
and `store.search(..., query=...)` finds the closest meanings.

(It uses your `OPENAI_API_KEY` — embedding calls cost a tiny fraction of a cent.)

In [ ]:
from uuid import uuid4
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.store.memory import InMemoryStore

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ★ index=... enables search by MEANING, not just by key
store = InMemoryStore(
    index={"dims": 1536, "embed": "openai:text-embedding-3-small"}
)
print("Setup done — store has semantic search enabled.")

## Step 2 — Save two past episodes (what worked)

Each episode is one structured story: **situation → response → result**.
We save them under the namespace `("agent", "support-bot", "episodes")` —
these memories belong to the *agent*, not to one user.

In [ ]:
episodes = [
    {"situation": "Customer angry about late delivery",
     "response": "Apologized sincerely, gave a 10% coupon, escalated to logistics",
     "result": "Customer satisfied, left a 5-star review"},
    {"situation": "Customer asked for a refund on a used item",
     "response": "Explained the policy politely, offered an exchange instead",
     "result": "Customer accepted the exchange"},
]

for ep in episodes:
    store.put(
        ("agent", "support-bot", "episodes"),
        key=str(uuid4()),   # unique key per episode
        value=ep,
    )

print(f"Saved {len(episodes)} episodes.")
print("
All episodes in the store:")
for item in store.search(("agent", "support-bot", "episodes")):
    print("  -", item.value["situation"])

## Step 3 — A new ticket arrives: recall similar episodes

The new ticket is **"My package arrived damaged and I want my money back."**

No words match our episodes exactly ("damaged" vs "late", "money back" vs "refund")...
but *meaning* matches. Watch semantic search find both relevant episodes.

In [ ]:
new_ticket = "My package arrived damaged and I want my money back"

similar = store.search(
    ("agent", "support-bot", "episodes"),
    query=new_ticket,   # ★ search by meaning
    limit=2,
)

print("Recalled episodes (most similar first):")
for m in similar:
    print(f"  - {m.value['situation']}  (score: {m.score:.2f})")

## Step 4 — Inject the episodes as few-shot guidance

Now we hand the recalled episodes to the LLM as *"here's how similar cases were
handled successfully"* — the agent answers the new ticket **in the spirit of what worked before**.

In [ ]:
few_shot = "

".join(
    f"Past situation: {m.value['situation']}
"
    f"What worked: {m.value['response']}
"
    f"Outcome: {m.value['result']}"
    for m in similar
)

answer = llm.invoke([
    SystemMessage(content=(
        "You are a support agent. Here is how similar cases were "
        f"handled successfully:

{few_shot}"
    )),
    HumanMessage(content=new_ticket),
])

print("AI reply to the new ticket:")
print(answer.content)

## Try it yourself

1. Add a third episode (e.g. *"customer couldn't log in → password reset link + video guide →
   solved in 5 minutes"*), then ask about *"I forgot my password"*. Does it get recalled?
2. Run the search **without** `query=new_ticket` — you get *all* episodes back, unranked.
   That difference (list everything vs. rank by meaning) is exactly why the `index=` line exists.
3. Discussion: who should write episodes into the store in a real app — a human reviewing
   tickets, or the agent itself after each success? What are the risks of each?

**So far:** semantic = what the agent **knows**, episodic = what the agent **did**.
**Next notebook:** the deepest one — memory that changes how the agent **behaves** → procedural.